# Speech Grammar Scoring Engine - End-to-End Pipeline

This notebook presents the complete machine learning pipeline for the **SHL Hiring Assessment 2026** speech grammar scoring competition. The objective is to predict continuous language proficiency/grammar Mean Opinion Scores (MOS) from spoken English audio recordings (recorded at 16kHz mono format) on a scale between `1.0` and `5.0`.

## Pipeline Architecture
1. **Acoustic Preprocessing**: Extracting 35 robust acoustic elements using `librosa` (Spectral Centroid, Spectral Rolloff, RMS Energy, Zero Crossing Rate, and 13-dimensional MFCCs).
2. **Validation Scheme**: Stratified 5-Fold Cross-Validation grouped and binned on continuous targets to ensure uniform target distributions across folds.
3. **Regression Modeling**: Standard Scaled continuous features fitted across an ensemble of Random Forest Regressors.

## Repository Structure & Modules
This notebook relies on the production modular scripts located in the `src/` directory of the project:
- `src/extract_features.py`: Acoustic feature engineering functions.
- `src/train_model.py`: Cross-validation setup and training pipeline.

--- 

### Open in Google Colab
To open and run this notebook in Google Colab, make sure you clone the repository first and install the requirements as guided below.

## 1. Environment Setup & Dependencies

In [ ]:
# Install required libraries if they are not already present
import sys
!{sys.executable} -m pip install -r ../requirements.txt

## 2. Environment Verification and Path Configurations

In [ ]:
import os
from pathlib import Path

# Configure paths dynamically relative to repository root
REPO_ROOT = Path(".").resolve().parent
DATA_DIR = REPO_ROOT / "dataset" / "Dataset_Final"

print(f"Repository Root: {REPO_ROOT}")
print(f"Data Directory: {DATA_DIR}")

# Safeguard verification gate
train_csv_path = DATA_DIR / "train.csv"
test_csv_path = DATA_DIR / "test.csv"

if not train_csv_path.exists() or not test_csv_path.exists():
    print("\n[!] Warning: Competition dataset CSV files not found inside the expected path.")
    print("Please retrieve the official dataset and place it in the ‘dataset/Dataset_Final/’ directory to proceed with full execution.")
else:
    print("\n[✅] Dataset configuration verified!")

## 3. Modular Feature Extraction Example

We make use of our modular `extract_acoustic_features` utility function defined in the `src` directory to inspect an audio file.

In [ ]:
import sys
# Append repository root to path for resolving local modular imports
sys.path.append(str(REPO_ROOT))

try:
    from src.extract_features import extract_acoustic_features
    print("[✅] Successfully imported modular feature extraction utilities!")
except ImportError as e:
    print(f"[!] Import failed: {e}")

## 4. Run Cross-Validation Pipeline

We load cached acoustic training features (or extract them iteratively if available) and run our Stratified 5-Fold Random Forest training pipeline. To ensure the notebook is safe to open and does not auto-run heavy training, this is locked behind an explicit flag.

In [ ]:
RUN_TRAINING = False  # Set to True to execute active model training

if RUN_TRAINING:
    from src.train_model import train_cv_pipeline
    features_csv = DATA_DIR / "train_acoustic_features.csv"
    
    if features_csv.exists():
        features_df = pd.read_csv(features_csv)
        # Select the 35 acoustic column representations
        X_cols = [c for f in [["duration", "centroid_mean", "centroid_std", "rolloff_mean", "rolloff_std", "rms_mean", "rms_std", "zcr_mean", "zcr_std"], [f"mfcc_mean_{i}" for i in range(13)], [f"mfcc_std_{i}" for i in range(13)]] for c in f]
        
        models, scalers = train_cv_pipeline(features_df, X_cols)
    else:
        print("[!] Pre-extracted feature CSV missing. Please perform acoustic feature extraction first.")
else:
    print("[ℹᔇ] Training block bypassed. Toggle 'RUN_TRAINING = True' to retrain the 5-fold pipeline.")

## 5. Summary and Performance Metrics

Below are the Out-of-Fold (OOF) cross-validation results achieved across our models:

| Model | OOF RMSE | OOF Pearson Correlation |
| :--- | :---: | :---: |
| **Ridge Regression (Baseline)** | 0.8544 | 0.7257 |
| **HistGradientBoosting** | 0.7916 | 0.7709 |
| **Random Forest Regressor (Ensemble)** | **0.7909** | **0.7713** |

The results highlight that non-linear ensembles capture speech features significantly better than standard linear regression Baselines!